# 1. Príprava dát

**Úloha:** rozpoznať z krátkeho videoklipu, či človek **je** (`eat`) alebo **pije** (`drink`).

Používame dataset **HMDB51** (51 činností, klipy z filmov a YouTube) vo verzii s už extrahovanými snímkami:
https://www.kaggle.com/datasets/easonlll/hmdb51

Z archívu stačí rozbaliť dve triedy:

```bash
unzip -q hmdb51.zip 'HMDB51/eat/*' 'HMDB51/drink/*' -d .
```

Štruktúra dát: `HMDB51/<trieda>/<klip>/<číslo snímky>.jpg`.

V tomto notebooku:
1. vytvoríme zoznam klipov (`index.csv`),
2. ukážeme, ako delíme dáta na trénovacie a testovacie,
3. pozrieme sa na dĺžku klipov a na ukážku snímok.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from har_utils import CLASSES, make_splits, read_frames

ROOT = Path("HMDB51")

## Zoznam klipov

Každý klip dostane:
- `label` – číslo triedy (0 = eat, 1 = drink),
- `source` – názov zdrojového videa (filmu). Je to časť názvu klipu pred názvom triedy,
  napr. `April_09_brush_hair_u_nm_np1_ba_goo_0` → `April_09`.

In [ ]:
rows = []
for label, cls in enumerate(CLASSES):
    for clip in sorted((ROOT / cls).iterdir()):
        if clip.is_dir():
            # názov klipu: <zdroj>_<trieda>_<6 polí metadát>
            source = clip.name.rsplit("_", 6)[0].removesuffix(f"_{cls}")
            rows.append((str(clip), cls, label, source))

df = pd.DataFrame(rows, columns=["path", "cls", "label", "source"])
df.to_csv("index.csv", index=False)

print(df.cls.value_counts())
print("počet zdrojových videí:", df.source.nunique())

## Delenie na trénovacie a testovacie dáta

Z jedného filmu je v HMDB51 často vystrihnutých viac klipov, ktoré sú si veľmi podobné.
Keby sme delili náhodne po klipoch, podobné zábery by sa dostali do tréningu aj do testu
a presnosť by bola nadhodnotená (**únik dát**).

Preto delíme **podľa zdrojového videa**: všetky klipy z jedného filmu sú buď v train, alebo v test.

Test má len okolo 80 klipov, takže jeden výsledok by bol veľmi zašumený
(jeden klip ≈ 1,25 %). Každý model preto hodnotíme na **10 rôznych deleniach** 70/30
a uvádzame priemer ± smerodajnú odchýlku. Funkcia `make_splits` je v `har_utils.py`.

Ukážka prvého delenia:

In [ ]:
seed, tr, te = next(make_splits(df))
part = pd.Series("train", index=df.index, name="split")
part.iloc[te] = "test"
print(pd.crosstab(part, df.cls))

## Dĺžka klipov

Z každého klipu neskôr vyberáme 16 rovnomerne rozložených snímok.

In [ ]:
n_frames = df.path.map(lambda p: len(list(Path(p).glob("*.jpg"))))
print(n_frames.describe().round(1))

plt.hist(n_frames, bins=30)
plt.xlabel("počet snímok v klipe")
plt.ylabel("počet klipov")
plt.show()

## Ukážka klipov

Takto vyzerá vstup modelu: 16 snímok rovnomerne vybraných z klipu.

In [ ]:
for cls in CLASSES:
    row = df[df.cls == cls].iloc[0]
    frames = read_frames(row.path)            # (16, 224, 224, 3)
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    for ax, f in zip(axes.flat, frames):
        ax.imshow(f)
        ax.axis("off")
    fig.suptitle(f"{cls}: {Path(row.path).name}")
    plt.show()